# Regression Loss Metrics and Monte Carlo Simulation

[Back to Session Overview](../README.md) · [Back to Module 01](../../README.md)

In supervised regression, the objective is to predict continuous quantities $y \in \mathbb{R}$. The loss function quantifies the discrepancy between ground-truth targets $y_i$ and model predictions $\hat{y}_i$. Different loss functions induce different optimization behaviors, sensitivities to outliers, and probabilistic interpretations.

### Objectives
1. Understand the mathematical properties of Absolute Error (AE) and Squared Error (SE).
2. Formulate Mean Absolute Error (MAE), Mean Squared Error (MSE), and Root Mean Squared Error (RMSE).
3. Implement a Monte Carlo simulation generating uniform random predictions and targets.
4. Observe sensitivity to large errors and compare MAE vs MSE robustness.


## 1. Mathematical Formulations

Given $n$ samples with targets $y_i$ and predictions $\hat{y}_i$:

1. **Absolute Error (AE) & Mean Absolute Error (MAE):**
   $$\text{AE}(y_i, \hat{y}_i) = |y_i - \hat{y}_i|$$
   $$\text{MAE} = \frac{1}{n} \sum_{i=1}^n |y_i - \hat{y}_i|$$
   - Penalty increases linearly with error magnitude.
   - Robust to outliers; represents the median estimate under Laplace noise.
   - Constant derivative $\pm 1$, non-differentiable at $y_i = \hat{y}_i$.

2. **Squared Error (SE) & Mean Squared Error (MSE):**
   $$\text{SE}(y_i, \hat{y}_i) = (y_i - \hat{y}_i)^2$$
   $$\text{MSE} = \frac{1}{n} \sum_{i=1}^n (y_i - \hat{y}_i)^2$$
   - Penalty increases quadratically with error magnitude.
   - Highly sensitive to outliers; represents the mean estimate under Gaussian noise.
   - Smooth and differentiable everywhere with gradient $2(\hat{y}_i - y_i)$.

3. **Root Mean Squared Error (RMSE):**
   $$\text{RMSE} = \sqrt{\text{MSE}} = \sqrt{\frac{1}{n} \sum_{i=1}^n (y_i - \hat{y}_i)^2}$$
   - Retains the quadratic penalty of MSE while expressing error in the original target units.


In [ ]:
import math
import random


def calc_ae(y: float, y_hat: float) -> float:
    """Calculates Absolute Error between target y and prediction y_hat."""
    return abs(y - y_hat)


def calc_se(y: float, y_hat: float) -> float:
    """Calculates Squared Error between target y and prediction y_hat."""
    return (y - y_hat) ** 2

## 2. Monte Carlo Simulation Function

The specification requires:
1. Validate `num_samples`: check `.isnumeric()`. If false, print `'number of samples must be an integer number'` and terminate.
2. Accept `loss_name`: `'MAE'`, `'MSE'`, or `'RMSE'`.
3. In a loop of `num_samples` iterations:
   - Generate predicted sample $\hat{y}_i \sim \text{Uniform}(0, 10)$.
   - Generate target sample $y_i \sim \text{Uniform}(0, 10)$.
   - Calculate step loss (AE for MAE; SE for MSE and RMSE).
   - Accumulate total loss.
   - Print per-sample trace: `loss_name, sample-i, pred, target, loss`.
4. Divide total loss by `num_samples`. If RMSE, compute `math.sqrt()`.
5. Print final loss.


In [ ]:
def simulate_regression_loss(num_samples_str: str, loss_name: str, seed: int = 42) -> None:
    """Runs a Monte Carlo regression loss simulation with random uniform samples."""
    if not num_samples_str.isnumeric():
        print("number of samples must be an integer number")
        return

    name = loss_name.strip().upper()
    valid_losses = {"MAE", "MSE", "RMSE"}
    if name not in valid_losses:
        print(f"loss name {loss_name} is not supported")
        return

    num_samples = int(num_samples_str)
    if num_samples <= 0:
        print("number of samples must be greater than zero")
        return

    random.seed(seed)
    total_loss = 0.0

    for i in range(num_samples):
        pred_sample = random.uniform(0.0, 10.0)
        target_sample = random.uniform(0.0, 10.0)

        if name == "MAE":
            step_loss = calc_ae(target_sample, pred_sample)
        else:  # MSE or RMSE accumulate squared error
            step_loss = calc_se(target_sample, pred_sample)

        total_loss += step_loss
        print(
            f"{name} sample-{i} | pred: {pred_sample:6.4f} | target: {target_sample:6.4f} | loss: {step_loss:7.4f}"
        )

    final_loss = total_loss / num_samples
    if name == "RMSE":
        final_loss = math.sqrt(final_loss)

    print(f"Final {name}: {final_loss:.6f}")

## 3. Running Simulations

We simulate MAE, MSE, and RMSE on 5 synthetic samples.


In [ ]:
print("=== Simulation: MAE with 5 samples ===")
simulate_regression_loss("5", "MAE", seed=10)

print("\n=== Simulation: MSE with 5 samples ===")
simulate_regression_loss("5", "MSE", seed=10)

print("\n=== Simulation: RMSE with 5 samples ===")
simulate_regression_loss("5", "RMSE", seed=10)

print("\n=== Error Handling: Non-numeric sample count ===")
simulate_regression_loss("five", "MAE")

## 4. Sensitivity Analysis: Robustness Against Outliers

Let us observe how MAE and MSE react when a single large outlier enters the dataset.


In [ ]:
clean_targets = [2.0, 3.0, 2.5, 3.5, 4.0]
clean_preds = [2.1, 2.9, 2.6, 3.4, 4.1]

# Corrupt with an extreme outlier
outlier_targets = [*clean_targets, 3.0]
outlier_preds = [*clean_preds, 25.0]  # Prediction is off by 22.0

mae_clean = sum(calc_ae(y, yh) for y, yh in zip(clean_targets, clean_preds, strict=True)) / len(
    clean_targets
)
mse_clean = sum(calc_se(y, yh) for y, yh in zip(clean_targets, clean_preds, strict=True)) / len(
    clean_targets
)

mae_outlier = sum(
    calc_ae(y, yh) for y, yh in zip(outlier_targets, outlier_preds, strict=True)
) / len(outlier_targets)
mse_outlier = sum(
    calc_se(y, yh) for y, yh in zip(outlier_targets, outlier_preds, strict=True)
) / len(outlier_targets)

print(f"Clean Data   -> MAE: {mae_clean:.4f} | MSE: {mse_clean:.4f}")
print(f"With Outlier -> MAE: {mae_outlier:.4f} | MSE: {mse_outlier:.4f}")
print(f"MAE Inflation Factor: {mae_outlier / mae_clean:.1f}x")
print(f"MSE Inflation Factor: {mse_outlier / mse_clean:.1f}x")